# The Lagrangian, FONCs, and the Second-Order Sufficient Condition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/DEV/L08/L08_lagrangian_and_sosc.ipynb)
### MECH 559 - Systems Optimization - L08

Companion notebook to the "Nonlinear programming" lecture. It reuses the ellipsoid-and-plane example from `L08_reduced_gradient.ipynb` to (1) show numerically that the Lagrangian FONCs and the reduced-gradient FONC are the *same* stationary points, just found a different way, and (2) apply the reduced-Hessian / SOSC test to tell the resulting minimizer and maximizer apart - something the FONCs alone cannot do.

## Learning objectives

By the end, students should be able to:

1. Form the Lagrangian $L(x,\lambda)=f(x)+\lambda^{\mathsf T}h(x)$ and state its FONCs $\nabla_xL=0$, $\nabla_\lambda L = h = 0$.
2. Verify numerically that $\lambda^{\mathsf T}=-\nabla_{x_s}f^{\mathsf T}(\nabla_{x_s}h)^{-1}$, i.e. that solving the Lagrangian system reproduces exactly the stationary points found by the reduced-gradient method.
3. Compute the Hessian of the Lagrangian $\nabla^2L$ and project it onto the tangent space to apply the SOSC.
4. Confirm numerically the identity $\partial x_d^{\mathsf T}\nabla^2w\,\partial x_d = \partial x^{\mathsf T}\nabla^2L\,\partial x$ from the slides, and use it to tell the minimizer from the maximizer.

> **Classroom rhythm:** pause at each **Predict** prompt, collect answers, then run the next cell.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import null_space
from scipy.optimize import fsolve

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"feasible": "#A8DADC", "boundary": "#00798C", "optimum": "#D1495B", "accent": "#EDAE49", "other": "#6C5B7B"}

print(f"NumPy {np.__version__}")

NumPy 2.5.2


---
## 1. Setup: the same ellipsoid-and-plane example

$$
\min_{x\in\mathbb{R}^3} f(x) = x_1^2+x_2^2+x_3^2
\quad\text{s.t.}\quad
h_1(x) = \frac{x_1^2}{4}+\frac{x_2^2}{5}+\frac{x_3^2}{25} - 1 = 0,
\qquad
h_2(x) = x_1+x_2-x_3 = 0.
$$

`L08_reduced_gradient.ipynb` found the nearest point ($f=4.4118$, a minimizer) and farthest point ($f=10$, a maximizer) on this ellipse by solving $\nabla w=0$ together with $h=0$. Here we set up the Lagrangian $L(x,\lambda)=f(x)+\lambda^{\mathsf T}h(x)$ instead and solve its FONCs directly.

In [2]:
def f(x):
    return x[0]**2 + x[1]**2 + x[2]**2

def h(x):
    return np.array([
        x[0]**2 / 4 + x[1]**2 / 5 + x[2]**2 / 25 - 1,
        x[0] + x[1] - x[2],
    ])

def grad_f(x):
    return np.array([2 * x[0], 2 * x[1], 2 * x[2]])

def jac_h(x):
    return np.array([
        [x[0] / 2, 2 * x[1] / 5, 2 * x[2] / 25],
        [1.0,       1.0,          -1.0],
    ])

def lagrangian_system(z):
    x, lam = z[:3], z[3:]
    stationarity = grad_f(x) + jac_h(x).T @ lam
    return np.concatenate([stationarity, h(x)])

rng = np.random.default_rng(559)
starts = np.column_stack([rng.uniform(-3, 3, size=(60, 3)), rng.uniform(-5, 5, size=(60, 2))])

sols = []
for z0 in starts:
    sol, info, ier, msg = fsolve(lagrangian_system, z0, full_output=True)
    if ier == 1 and np.linalg.norm(h(sol[:3])) < 1e-8:
        if not any(np.allclose(sol[:3], s[:3], atol=1e-4) for s in sols):
            sols.append(sol)

sols.sort(key=lambda z: f(z[:3]))
print(f"found {len(sols)} distinct Lagrangian stationary points:\n")
for i, z in enumerate(sols):
    x, lam = z[:3], z[3:]
    print(f"solution {i}: x = {x}, lambda = {lam}, f(x) = {f(x):.6f}")

found 4 distinct Lagrangian stationary points:

solution 0: x = [-1.57377895  1.37705658 -0.19672237], lambda = [-4.41176471 -0.32401331], f(x) = 4.411765
solution 1: x = [ 1.57377895 -1.37705658  0.19672237], lambda = [-4.41176471  0.32401331], f(x) = 4.411765
solution 2: x = [1.02597835 1.53896753 2.56494588], lambda = [-10.           3.07793506], f(x) = 10.000000
solution 3: x = [-1.02597835 -1.53896753 -2.56494588], lambda = [-10.          -3.07793506], f(x) = 10.000000


---
## 2. Cross-check: reduced-gradient $\lambda$ vs. Lagrangian $\lambda$

The reduced-gradient derivation showed $\lambda^{\mathsf T} = -\nabla_{x_s}f^{\mathsf T}(\nabla_{x_s}h)^{-1}$ (state variables $x_1,x_2$; decision variable $x_3$). We compute that formula directly at each solution and compare it against the $\lambda$ the 5-variable Lagrangian solve just returned - they should agree exactly, since they are two derivations of the *same* FONC.

In [3]:
def lambda_from_reduced(x):
    J = jac_h(x)
    Js, Jd = J[:, :2], J[:, 2:]
    gfs = grad_f(x)[:2]
    return -gfs @ np.linalg.inv(Js)

for i, z in enumerate(sols):
    x, lam_lagrangian = z[:3], z[3:]
    lam_reduced = lambda_from_reduced(x)
    print(f"solution {i}: lambda (Lagrangian solve) = {lam_lagrangian}")
    print(f"             lambda (reduced-gradient)  = {lam_reduced}")
    print(f"             match? {np.allclose(lam_lagrangian, lam_reduced, atol=1e-4)}\n")

x_min, lam_min = sols[0][:3], sols[0][3:]
x_max, lam_max = sols[-1][:3], sols[-1][3:]

solution 0: lambda (Lagrangian solve) = [-4.41176471 -0.32401331]
             lambda (reduced-gradient)  = [-4.41176471 -0.32401331]
             match? True

solution 1: lambda (Lagrangian solve) = [-4.41176471  0.32401331]
             lambda (reduced-gradient)  = [-4.41176471  0.32401331]
             match? True

solution 2: lambda (Lagrangian solve) = [-10.           3.07793506]
             lambda (reduced-gradient)  = [-10.           3.07793506]
             match? True

solution 3: lambda (Lagrangian solve) = [-10.          -3.07793506]
             lambda (reduced-gradient)  = [-10.          -3.07793506]
             match? True



---
## 3. FONCs are not enough: apply the SOSC

Both points found above are stationary - the FONCs $\nabla_xL=0$, $h=0$ hold at *both*. To tell the minimizer from the maximizer we need the reduced (constrained) Hessian, using the slide's identity

$$
\partial x_d^{\mathsf T}\nabla^2w\,\partial x_d = \partial x^{\mathsf T}\nabla^2L\,\partial x
\qquad\text{for } \partial x \text{ in the tangent hyperplane.}
$$

> **Predict:** before running the next cell, guess the sign of $\partial x^{\mathsf T}\nabla^2L\,\partial x$ (restricted to the tangent direction) at the $f=4.41$ point, and at the $f=10$ point.

In [4]:
def hess_f(x):
    return 2 * np.eye(3)

def hess_h1(x):
    return np.diag([1 / 2, 2 / 5, 2 / 25])   # h1 = x1^2/4 + x2^2/5 + x3^2/25 - 1

def hess_h2(x):
    return np.zeros((3, 3))                    # h2 is linear

def hess_lagrangian(x, lam):
    return hess_f(x) + lam[0] * hess_h1(x) + lam[1] * hess_h2(x)

def sosc_check(x, lam, label):
    J = jac_h(x)
    y = null_space(J)[:, 0]          # 1D tangent direction (p = n - m = 1 here)
    y = y / y[2]                      # normalize so the decision-variable component is 1
    HL = hess_lagrangian(x, lam)
    quad_form = y @ HL @ y
    verdict = "positive definite -> local MINIMIZER" if quad_form > 0 else "negative definite -> local MAXIMIZER"
    print(f"{label}: x = {x}")
    print(f"  tangent direction (normalized) y = {y}")
    print(f"  y^T (nabla^2 L) y = {quad_form:.6f}  -> {verdict}\n")
    return quad_form

q_min = sosc_check(x_min, lam_min, "f = 4.41 point")
q_max = sosc_check(x_max, lam_max, "f = 10 point")

f = 4.41 point: x = [-1.57377895  1.37705658 -0.19672237]
  tangent direction (normalized) y = [0.4 0.6 1. ]
  y^T (nabla^2 L) y = 1.698824  -> positive definite -> local MINIMIZER

f = 10 point: x = [-1.02597835 -1.53896753 -2.56494588]
  tangent direction (normalized) y = [ 8. -7.  1.]
  y^T (nabla^2 L) y = -288.800000  -> negative definite -> local MAXIMIZER



---
## 4. Verifying the identity against a finite-difference reduced Hessian

As an independent check, we compute $\nabla^2w$ directly by finite-differencing $w(x_3)=f(x_s(x_3),x_3)$ (solving $h=0$ for $x_s$ at each nearby $x_3$) and compare its second derivative to the projected $\nabla^2L$ quadratic form from Section 3.

In [5]:
def state_for_decision(x3, xs_guess):
    def resid(xs):
        return h(np.array([xs[0], xs[1], x3]))
    return fsolve(resid, xs_guess, full_output=False)

def w_of_x3(x3, xs_guess):
    xs = state_for_decision(x3, xs_guess)
    return f(np.array([*xs, x3])), xs

def finite_diff_reduced_hessian(x_sol, eps=1e-4):
    x3_0 = x_sol[2]
    xs_guess = x_sol[:2]
    w0, _ = w_of_x3(x3_0, xs_guess)
    w_plus, xs_plus = w_of_x3(x3_0 + eps, xs_guess)
    w_minus, xs_minus = w_of_x3(x3_0 - eps, xs_guess)
    return (w_plus - 2 * w0 + w_minus) / eps**2

w2_min = finite_diff_reduced_hessian(x_min)
w2_max = finite_diff_reduced_hessian(x_max)

print(f"f = 4.41 point: finite-difference nabla^2 w = {w2_min:.6f}   vs.  projected nabla^2 L = {q_min:.6f}")
print(f"f = 10   point: finite-difference nabla^2 w = {w2_max:.6f}   vs.  projected nabla^2 L = {q_max:.6f}")
print(f"\nmatch (min)? {np.isclose(w2_min, q_min, rtol=1e-3)}")
print(f"match (max)? {np.isclose(w2_max, q_max, rtol=1e-3)}")

f = 4.41 point: finite-difference nabla^2 w = 1.698824   vs.  projected nabla^2 L = 1.698824
f = 10   point: finite-difference nabla^2 w = -288.815859   vs.  projected nabla^2 L = -288.800000

match (min)? True
match (max)? True


The finite-difference reduced Hessian $\nabla^2w$ and the tangent-space-projected $\nabla^2L$ agree (the $f=10$ point needs a looser tolerance since $\nabla_{x_s}h$ is more ill-conditioned there, as noted in the reduced-gradient notebook - finite differences amplify that). Both confirm: the $f=4.4118$ point is a genuine local **minimizer** (positive curvature), and the $f=10$ point is a local **maximizer** (negative curvature) - exactly matching the direct-NLP cross-check from `L08_reduced_gradient.ipynb`.

---
## Takeaways

1. The Lagrangian FONCs ($\nabla_xL=0$, $h=0$) and the reduced-gradient FONC ($\nabla w=0$, $h=0$) are algebraically equivalent - solving either one numerically lands on the same stationary points, and $\lambda$ from the Lagrangian solve matches $-\nabla_{x_s}f^{\mathsf T}(\nabla_{x_s}h)^{-1}$ from the reduced-gradient derivation to solver tolerance.
2. Both stationary points found here satisfy the FONCs equally well - the FONCs alone cannot distinguish the minimizer from the maximizer.
3. The reduced Hessian test - equivalently, projecting $\nabla^2L$ onto the tangent hyperplane - resolves that ambiguity: positive definite there means a local minimizer, negative definite means a local maximizer.
4. This is exactly the machinery the KKT conditions build on for *inequality* constraints too, covered next in `L08_kkt_conditions.ipynb` - including a worked example (Class Exercise 3, Q1) where a KKT point turns out to be a maximizer, not a minimizer, for exactly this reason.

## Optional exercises

1. Redo Section 3's SOSC check using $x_2,x_3$ as the tangent-normalizing component instead of $x_3$ (i.e. normalize $y$ so $y[1]=1$). Confirm the sign of the quadratic form is unchanged (only its value scales).
2. The Lagrangian multistart in Section 1 searches over $(x,\lambda)\in\mathbb{R}^3\times\mathbb{R}^2$. Try tightening or loosening the random-start bounds and see whether you ever find a *third* distinct stationary point - is that consistent with the geometry of an ellipse-shaped feasible curve?
3. Perturb $h_1$ slightly (e.g. change the $25$ to $16$) and repeat the whole notebook. Do the qualitative conclusions (one minimizer, one maximizer, opposite-sign reduced Hessians) still hold?